# **Data Preprocessing**

**Initial Exploratory Data Analysis**

Loading dataset

In [47]:
import pandas as pd
df = pd.read_csv('../data/processed/feedback_data_sentiment.csv')

Dataset Inspection

In [48]:
df.head()

,RespondentID,Feedback,Sentiment
0,1,Super smooth taste. imo it's one of the better...,Positive
1,2,Taste is kinda weak. Please introduce more roa...,Neutral
2,3,Expected better. I drink it every morning.,Neutral
3,4,"Love the aroma, not a fan of the aftertaste. n...",Neutral
4,5,Worth every penny. ngl I expected more. 😕,Neutral


In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   RespondentID  250 non-null    int64
 1   Feedback      250 non-null    str  
 2   Sentiment     250 non-null    str  
dtypes: int64(1), str(2)
memory usage: 19.6 KB


In [50]:
df.describe()

,RespondentID
count,250.000000
mean,125.500000
std,72.312977
min,1.000000
25%,63.250000
50%,125.500000
75%,187.750000
max,250.000000


In [51]:
df.isnull().sum()

RespondentID    0
Feedback        0
Sentiment       0
dtype: int64

In [52]:
df.duplicated().sum()

np.int64(0)

Rows: 1500  
Columns: 2  
Missing values: None  
Duplicated values: None  
  
Since no missing or duplicated values, no cleaning via 'drop' required. 

**Text Cleaning**

* Slang/abbreviations
* Lowercasing
* Removing URLs
* Removing HTML
* Removing emojis
* Removing punctiations
* Removing numbers
* Removing extra spacing

In [53]:
import re
import string
import emoji

slang_dict = {
    "ngl": "honestly", #tokenizing not gonna lie >>>'not gon na lie' then lemmmatizing would result in loss of information
    "tbh": "honestly",#similarly here, so a single word summary is used instead
    "imo": "opinion",
    "idk": "unknown"
}

def expand_slang(text):
    for word, replacement in slang_dict.items():
        text = re.sub(
            rf"\b{word}\b",
            replacement,
            text
        )
    return text

def clean_text(text):
    # Lowercasing
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML
    text = re.sub(r"<.*?>", "", text)

    # Remove emojis
    text = emoji.replace_emoji(text, replace="")

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text

In [54]:
df["clean_feedback"] = (
    df["Feedback"]
    .apply(expand_slang)
    .apply(clean_text)
)

In [55]:
df.head(100)

,RespondentID,Feedback,Sentiment,clean_feedback
0,1,Super smooth taste. imo it's one of the better...,Positive,super smooth taste opinion its one of the bett...
1,2,Taste is kinda weak. Please introduce more roa...,Neutral,taste is kinda weak please introduce more roas...
2,3,Expected better. I drink it every morning.,Neutral,expected better i drink it every morning
3,4,"Love the aroma, not a fan of the aftertaste. n...",Neutral,love the aroma not a fan of the aftertaste hon...
4,5,Worth every penny. ngl I expected more. 😕,Neutral,worth every penny honestly i expected more
...,...,...,...,...
95,96,Really enjoying Coffee X.,Positive,really enjoying coffee x
96,97,Wouldn't buy again,Negative,wouldnt buy again
97,98,Too expensive 😕,Negative,too expensive
98,99,realy enjoying Coffee X. Please introduce more...,Positive,realy enjoying coffee x please introduce more ...


In [56]:
df.to_csv(
    "../data/processed/clean_feedback_data_sentiment.csv",
    index=False
)

**NLP Preprocessing**

* Tokenization
* Stopword removal
* Lemmetization

In [57]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kowoy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kowoy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kowoy\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\kowoy\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [58]:
#Loading new cleaned dataset
df = pd.read_csv('../data/processed/clean_feedback_data_sentiment.csv')

In [59]:
df["clean_feedback"].isna().sum()

df = df.dropna(subset=["clean_feedback"])

df = df[df["clean_feedback"].str.strip() != ""]

In [60]:
#Tokenization
df["tokens"] = df["clean_feedback"].apply(word_tokenize)
df.head(100)

,RespondentID,Feedback,Sentiment,clean_feedback,tokens
0,1,Super smooth taste. imo it's one of the better...,Positive,super smooth taste opinion its one of the bett...,"[super, smooth, taste, opinion, its, one, of, ..."
1,2,Taste is kinda weak. Please introduce more roa...,Neutral,taste is kinda weak please introduce more roas...,"[taste, is, kinda, weak, please, introduce, mo..."
2,3,Expected better. I drink it every morning.,Neutral,expected better i drink it every morning,"[expected, better, i, drink, it, every, morning]"
3,4,"Love the aroma, not a fan of the aftertaste. n...",Neutral,love the aroma not a fan of the aftertaste hon...,"[love, the, aroma, not, a, fan, of, the, after..."
4,5,Worth every penny. ngl I expected more. 😕,Neutral,worth every penny honestly i expected more,"[worth, every, penny, honestly, i, expected, m..."
...,...,...,...,...,...
97,98,Too expensive 😕,Negative,too expensive,"[too, expensive]"
98,99,realy enjoying Coffee X. Please introduce more...,Positive,realy enjoying coffee x please introduce more ...,"[realy, enjoying, coffee, x, please, introduce..."
99,100,Expected better. Not bad at all tbh. ☕,Neutral,expected better not bad at all honestly,"[expected, better, not, bad, at, all, honestly]"
100,101,Worth every penny. I drink it every morning.,Positive,worth every penny i drink it every morning,"[worth, every, penny, i, drink, it, every, mor..."


In [61]:
#Stopword removal
#Creating stopword list
stop_words = set(stopwords.words("english"))

#Keeping relevant sentiment words
stop_words = stop_words - {
    "not",
    "no",
    "nor",
    "never"
}

#Creating stopword removal function
def remove_stopwords(tokens):

    filtered_tokens = [
        word for word in tokens
        if word.lower() not in stop_words
    ]

    return filtered_tokens

#Apply stopword removal function within new column 'filtered_tokens'
df["filtered_tokens"] = df["tokens"].apply(remove_stopwords)

In [62]:
df.head(100)

,RespondentID,Feedback,Sentiment,clean_feedback,tokens,filtered_tokens
0,1,Super smooth taste. imo it's one of the better...,Positive,super smooth taste opinion its one of the bett...,"[super, smooth, taste, opinion, its, one, of, ...","[super, smooth, taste, opinion, one, better, b..."
1,2,Taste is kinda weak. Please introduce more roa...,Neutral,taste is kinda weak please introduce more roas...,"[taste, is, kinda, weak, please, introduce, mo...","[taste, kinda, weak, please, introduce, roast,..."
2,3,Expected better. I drink it every morning.,Neutral,expected better i drink it every morning,"[expected, better, i, drink, it, every, morning]","[expected, better, drink, every, morning]"
3,4,"Love the aroma, not a fan of the aftertaste. n...",Neutral,love the aroma not a fan of the aftertaste hon...,"[love, the, aroma, not, a, fan, of, the, after...","[love, aroma, not, fan, aftertaste, honestly, ..."
4,5,Worth every penny. ngl I expected more. 😕,Neutral,worth every penny honestly i expected more,"[worth, every, penny, honestly, i, expected, m...","[worth, every, penny, honestly, expected]"
...,...,...,...,...,...,...
97,98,Too expensive 😕,Negative,too expensive,"[too, expensive]",[expensive]
98,99,realy enjoying Coffee X. Please introduce more...,Positive,realy enjoying coffee x please introduce more ...,"[realy, enjoying, coffee, x, please, introduce...","[realy, enjoying, coffee, x, please, introduce..."
99,100,Expected better. Not bad at all tbh. ☕,Neutral,expected better not bad at all honestly,"[expected, better, not, bad, at, all, honestly]","[expected, better, not, bad, honestly]"
100,101,Worth every penny. I drink it every morning.,Positive,worth every penny i drink it every morning,"[worth, every, penny, i, drink, it, every, mor...","[worth, every, penny, drink, every, morning]"


In [63]:
#Create lemmatizer
lemmatizer = WordNetLemmatizer()
#Create lemmatizer function
def lemmatize_tokens(tokens):
    lemmatized = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]
    return lemmatized

df["lemmatized_tokens"] = df["filtered_tokens"].apply(lemmatize_tokens)

In [64]:
df.head(100)

,RespondentID,Feedback,Sentiment,clean_feedback,tokens,filtered_tokens,lemmatized_tokens
0,1,Super smooth taste. imo it's one of the better...,Positive,super smooth taste opinion its one of the bett...,"[super, smooth, taste, opinion, its, one, of, ...","[super, smooth, taste, opinion, one, better, b...","[super, smooth, taste, opinion, one, better, b..."
1,2,Taste is kinda weak. Please introduce more roa...,Neutral,taste is kinda weak please introduce more roas...,"[taste, is, kinda, weak, please, introduce, mo...","[taste, kinda, weak, please, introduce, roast,...","[taste, kinda, weak, please, introduce, roast,..."
2,3,Expected better. I drink it every morning.,Neutral,expected better i drink it every morning,"[expected, better, i, drink, it, every, morning]","[expected, better, drink, every, morning]","[expected, better, drink, every, morning]"
3,4,"Love the aroma, not a fan of the aftertaste. n...",Neutral,love the aroma not a fan of the aftertaste hon...,"[love, the, aroma, not, a, fan, of, the, after...","[love, aroma, not, fan, aftertaste, honestly, ...","[love, aroma, not, fan, aftertaste, honestly, ..."
4,5,Worth every penny. ngl I expected more. 😕,Neutral,worth every penny honestly i expected more,"[worth, every, penny, honestly, i, expected, m...","[worth, every, penny, honestly, expected]","[worth, every, penny, honestly, expected]"
...,...,...,...,...,...,...,...
97,98,Too expensive 😕,Negative,too expensive,"[too, expensive]",[expensive],[expensive]
98,99,realy enjoying Coffee X. Please introduce more...,Positive,realy enjoying coffee x please introduce more ...,"[realy, enjoying, coffee, x, please, introduce...","[realy, enjoying, coffee, x, please, introduce...","[realy, enjoying, coffee, x, please, introduce..."
99,100,Expected better. Not bad at all tbh. ☕,Neutral,expected better not bad at all honestly,"[expected, better, not, bad, at, all, honestly]","[expected, better, not, bad, honestly]","[expected, better, not, bad, honestly]"
100,101,Worth every penny. I drink it every morning.,Positive,worth every penny i drink it every morning,"[worth, every, penny, i, drink, it, every, mor...","[worth, every, penny, drink, every, morning]","[worth, every, penny, drink, every, morning]"


In [65]:
#Final column creation where lemmatized tokens are joined into a single string
df["processed_text"] = df["lemmatized_tokens"].apply(
    lambda x: " ".join(x)
)
df.head(100)

,RespondentID,Feedback,Sentiment,clean_feedback,tokens,filtered_tokens,lemmatized_tokens,processed_text
0,1,Super smooth taste. imo it's one of the better...,Positive,super smooth taste opinion its one of the bett...,"[super, smooth, taste, opinion, its, one, of, ...","[super, smooth, taste, opinion, one, better, b...","[super, smooth, taste, opinion, one, better, b...",super smooth taste opinion one better brand
1,2,Taste is kinda weak. Please introduce more roa...,Neutral,taste is kinda weak please introduce more roas...,"[taste, is, kinda, weak, please, introduce, mo...","[taste, kinda, weak, please, introduce, roast,...","[taste, kinda, weak, please, introduce, roast,...",taste kinda weak please introduce roast option
2,3,Expected better. I drink it every morning.,Neutral,expected better i drink it every morning,"[expected, better, i, drink, it, every, morning]","[expected, better, drink, every, morning]","[expected, better, drink, every, morning]",expected better drink every morning
3,4,"Love the aroma, not a fan of the aftertaste. n...",Neutral,love the aroma not a fan of the aftertaste hon...,"[love, the, aroma, not, a, fan, of, the, after...","[love, aroma, not, fan, aftertaste, honestly, ...","[love, aroma, not, fan, aftertaste, honestly, ...",love aroma not fan aftertaste honestly expected
4,5,Worth every penny. ngl I expected more. 😕,Neutral,worth every penny honestly i expected more,"[worth, every, penny, honestly, i, expected, m...","[worth, every, penny, honestly, expected]","[worth, every, penny, honestly, expected]",worth every penny honestly expected
...,...,...,...,...,...,...,...,...
97,98,Too expensive 😕,Negative,too expensive,"[too, expensive]",[expensive],[expensive],expensive
98,99,realy enjoying Coffee X. Please introduce more...,Positive,realy enjoying coffee x please introduce more ...,"[realy, enjoying, coffee, x, please, introduce...","[realy, enjoying, coffee, x, please, introduce...","[realy, enjoying, coffee, x, please, introduce...",realy enjoying coffee x please introduce roast...
99,100,Expected better. Not bad at all tbh. ☕,Neutral,expected better not bad at all honestly,"[expected, better, not, bad, at, all, honestly]","[expected, better, not, bad, honestly]","[expected, better, not, bad, honestly]",expected better not bad honestly
100,101,Worth every penny. I drink it every morning.,Positive,worth every penny i drink it every morning,"[worth, every, penny, i, drink, it, every, mor...","[worth, every, penny, drink, every, morning]","[worth, every, penny, drink, every, morning]",worth every penny drink every morning


In [66]:
#Export new csv with lemmatized text responses
df.to_csv(
    "lemmatized_reviews_sentiment.csv",
    index=False,
    encoding="utf-8"
)

**TF-IDF Vectorization**

In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)